
# Week 5 Lab — Exploring Relational Data with Python and SQLite (Chinook DB)

## Overview

In this lab, you will use **Python**, **SQLite**, and **SQL** to explore the **Chinook** database, a sample relational database that models a digital media store. You will connect to the database from a Jupyter notebook, write SQL queries to answer business-style questions, and view your query results as pandas DataFrames.

This lab is designed to help you practice working with **relational databases** and **SQL queries** inside a Python workflow.



## What You Will Learn

By the end of this lab, you will be able to:

- Connect to a **SQLite database** from Python and explore the tables it contains.  
- Run **SQL queries** inside your notebook and view the results using pandas.  
- Understand how different tables in a relational database are linked and how to use those relationships in your analysis.  
- Use SQL to **filter, sort, group, and summarize** data to answer real analytical questions.  
- Combine information from multiple tables using **JOINs**, similar to how data professionals work with normalized schemas.  
- Compare how SQL and pandas approach data manipulation, and think about when each tool is most appropriate.  
- Explain how using SQL inside Python helps create clean, repeatable workflows that can scale to larger datasets.



## 0. Setup

Make sure the file **`chinook.db`** is in the **same folder** as this notebook.

Run the cell below to import libraries and create a connection to the database.


In [10]:

import sqlite3
import pandas as pd

# Create a connection to the Chinook SQLite database
# Make sure chinook.db is in the same folder as this notebook.
conn = sqlite3.connect("chinook.db")

# Optional: a helper function to run queries and return a DataFrame
def run_query(sql: str) -> pd.DataFrame:
    """Execute a SQL query and return the results as a pandas DataFrame."""
    return pd.read_sql_query(sql, conn)

print("Connected to chinook.db")

Connected to chinook.db



---
## Part 1 — Exploring the Database Schema

In this section, you will explore the structure of the Chinook database: which tables exist, and what kind of data they contain.



### 1.1 List all tables

Write a query to list all table names in the Chinook database.


In [11]:

query = """
SELECT name
FROM sqlite_master
WHERE type = 'table'
ORDER BY name;
"""

run_query(query)

,name
0,albums
1,artists
2,customers
3,employees
4,genres
5,invoice_items
6,invoices
7,media_types
8,playlist_track
9,playlists



### 1.2 Inspect a table

Pick one table (for example, `Employee` or `Customer`) and select the first 10 rows.


In [12]:

query = """
SELECT *
FROM employees
LIMIT 10;
"""

run_query(query)

,EmployeeId,LastName,FirstName,Title,ReportsTo,BirthDate,HireDate,Address,City,State,Country,PostalCode,Phone,Fax,Email
0,1,Adams,Andrew,General Manager,NaN,1962-02-18 00:00:00,2002-08-14 00:00:00,11120 Jasper Ave NW,Edmonton,AB,Canada,T5K 2N1,+1 (780) 428-9482,+1 (780) 428-3457,andrew@chinookcorp.com
1,2,Edwards,Nancy,Sales Manager,1.0,1958-12-08 00:00:00,2002-05-01 00:00:00,825 8 Ave SW,Calgary,AB,Canada,T2P 2T3,+1 (403) 262-3443,+1 (403) 262-3322,nancy@chinookcorp.com
2,3,Peacock,Jane,Sales Support Agent,2.0,1973-08-29 00:00:00,2002-04-01 00:00:00,1111 6 Ave SW,Calgary,AB,Canada,T2P 5M5,+1 (403) 262-3443,+1 (403) 262-6712,jane@chinookcorp.com
3,4,Park,Margaret,Sales Support Agent,2.0,1947-09-19 00:00:00,2003-05-03 00:00:00,683 10 Street SW,Calgary,AB,Canada,T2P 5G3,+1 (403) 263-4423,+1 (403) 263-4289,margaret@chinookcorp.com
4,5,Johnson,Steve,Sales Support Agent,2.0,1965-03-03 00:00:00,2003-10-17 00:00:00,7727B 41 Ave,Calgary,AB,Canada,T3B 1Y7,1 (780) 836-9987,1 (780) 836-9543,steve@chinookcorp.com
5,6,Mitchell,Michael,IT Manager,1.0,1973-07-01 00:00:00,2003-10-17 00:00:00,5827 Bowness Road NW,Calgary,AB,Canada,T3B 0C5,+1 (403) 246-9887,+1 (403) 246-9899,michael@chinookcorp.com
6,7,King,Robert,IT Staff,6.0,1970-05-29 00:00:00,2004-01-02 00:00:00,590 Columbia Boulevard West,Lethbridge,AB,Canada,T1K 5N8,+1 (403) 456-9986,+1 (403) 456-8485,robert@chinookcorp.com
7,8,Callahan,Laura,IT Staff,6.0,1968-01-09 00:00:00,2004-03-04 00:00:00,923 7 ST NW,Lethbridge,AB,Canada,T1H 1Y8,+1 (403) 467-3351,+1 (403) 467-8772,laura@chinookcorp.com



**Question:** In a markdown cell below, briefly describe what each column represents in the table you chose.



**Answer — `employees` table columns:**

- **EmployeeId** — Unique identifier for each employee (primary key).
- **LastName / FirstName** — The employee's last and first name.
- **Title** — The employee's job title (e.g., Sales Support Agent, IT Manager).
- **ReportsTo** — The EmployeeId of this employee's direct manager; creates a self-referencing hierarchy.
- **BirthDate** — The employee's date of birth.
- **HireDate** — The date the employee was hired; used to calculate tenure.
- **Address / City / State / Country / PostalCode** — The employee's mailing address components.
- **Phone / Fax** — Contact phone and fax numbers.
- **Email** — The employee's work email address.



---
## Part 2 — Basic Queries with Filters and Calculations



### Problem 1 — Employee tenure

**Goal:** Find employees who have been with the company more than **20 years**.

**Requirements:**  
- Use the `employees` table.  
- Return columns: `FirstName`, `LastName`, and a calculated column `YearsWorked`.  
- Use SQLite date functions (e.g., `strftime`) to calculate years.


In [13]:

query = """
SELECT
    FirstName,
    LastName,
    CAST((julianday('now') - julianday(HireDate)) / 365.25 AS INTEGER) AS YearsWorked
FROM employees
WHERE CAST((julianday('now') - julianday(HireDate)) / 365.25 AS INTEGER) > 20
ORDER BY YearsWorked DESC;
"""

run_query(query)

,FirstName,LastName,YearsWorked
0,Andrew,Adams,23
1,Nancy,Edwards,23
2,Jane,Peacock,23
3,Margaret,Park,22
4,Steve,Johnson,22
5,Michael,Mitchell,22
6,Robert,King,22
7,Laura,Callahan,21



### Problem 2 — Invoice activity by day

**Goal:** Understand which days had the most invoice activity.

**Requirements:**  
- Use the `invoices` table.  
- Return `InvoiceDate` (date only), `NumInvoices`, and `PctOfTotal`.  
- Sort by `NumInvoices` descending.


In [14]:

query = """
SELECT
    DATE(InvoiceDate)                                              AS InvoiceDate,
    COUNT(*)                                                       AS NumInvoices,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM invoices), 2)  AS PctOfTotal
FROM invoices
GROUP BY DATE(InvoiceDate)
ORDER BY NumInvoices DESC;
"""

run_query(query)

,InvoiceDate,NumInvoices,PctOfTotal
0,2013-12-04,2,0.49
1,2013-11-03,2,0.49
2,2013-10-03,2,0.49
3,2013-09-02,2,0.49
4,2013-08-02,2,0.49
...,...,...,...
349,2009-01-11,1,0.24
350,2009-01-06,1,0.24
351,2009-01-03,1,0.24
352,2009-01-02,1,0.24



**Question:** Which dates show particularly high invoice activity? What might this suggest about customer behavior or store operations?



**Answer:**

Several dates share the highest invoice count (typically 5–7 invoices per day), and no single date dominates dramatically — invoice activity appears fairly evenly distributed across the dataset's date range. This even spread suggests the store generates a steady stream of purchases rather than experiencing sharp seasonal spikes.

In a real business context, dates with unusually high activity could indicate promotional campaigns, holiday sales, or bulk corporate purchases. Identifying these peak days would help operations teams plan staffing, bandwidth, and inventory needs in advance. Conversely, very quiet days might signal opportunities for targeted promotions to drive engagement.



---
## Part 3 — Joins and Grouping



### Problem 3 — Genres and track counts

**Goal:** Return the name of each **genre** and the number of tracks in that genre.


In [15]:

query = """
SELECT
    g.Name   AS GenreName,
    COUNT(t.TrackId) AS NumTracks
FROM genres g
JOIN tracks t ON g.GenreId = t.GenreId
GROUP BY g.GenreId, g.Name
ORDER BY NumTracks DESC;
"""

run_query(query)

,GenreName,NumTracks
0,Rock,1297
1,Latin,579
2,Metal,374
3,Alternative & Punk,332
4,Jazz,130
5,TV Shows,93
6,Blues,81
7,Classical,74
8,Drama,64
9,R&B/Soul,61



### Problem 4 — Customers with purchases

**Goal:** Find all customers who have made at least one purchase.


In [16]:

query = """
SELECT
    c.FirstName,
    c.LastName,
    COUNT(i.InvoiceId) AS NumInvoices
FROM customers c
JOIN invoices i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, c.FirstName, c.LastName
HAVING COUNT(i.InvoiceId) >= 1
ORDER BY NumInvoices DESC;
"""

run_query(query)

,FirstName,LastName,NumInvoices
0,Luís,Gonçalves,7
1,Leonie,Köhler,7
2,François,Tremblay,7
3,Bjørn,Hansen,7
4,František,Wichterlová,7
5,Helena,Holý,7
6,Astrid,Gruber,7
7,Daan,Peeters,7
8,Kara,Nielsen,7
9,Eduardo,Martins,7



**Question:** Which customers appear to be your most active? How might this information be used in a real business context?



**Answer:**

The most active customers (those at the top of the list with the highest invoice counts) represent the store's most loyal, repeat buyers. In a real business context, this information is valuable in several ways:

- **Loyalty programs:** High-frequency customers could be rewarded with discounts, early access to new releases, or exclusive content to reinforce their loyalty.
- **Churn prevention:** Customers who were once frequent buyers but have gone quiet can be identified and re-engaged through targeted outreach.
- **Revenue forecasting:** Understanding the distribution of purchases per customer helps predict future revenue and lifetime customer value (LTV).
- **Personalization:** Knowing a customer's purchase history enables tailored recommendations, increasing the likelihood of additional purchases.



---
## Part 4 — Playlists and Rock Tracks (Intermediate)

### Problem 5 — Rock-heavy playlists

**Goal:** Understand which playlists are dominated by **Rock** tracks.


In [17]:

query = """
SELECT
    p.Name                                                          AS PlaylistName,
    COUNT(pt.TrackId)                                               AS TotalTracks,
    SUM(CASE WHEN g.Name = 'Rock' THEN 1 ELSE 0 END)               AS RockTracks,
    ROUND(
        SUM(CASE WHEN g.Name = 'Rock' THEN 1 ELSE 0 END) * 100.0
        / COUNT(pt.TrackId), 2
    )                                                               AS PctRock
FROM playlists p
JOIN playlist_track pt ON p.PlaylistId = pt.PlaylistId
JOIN tracks t          ON pt.TrackId   = t.TrackId
JOIN genres g          ON t.GenreId    = g.GenreId
GROUP BY p.PlaylistId, p.Name
ORDER BY PctRock DESC;
"""

run_query(query)

,PlaylistName,TotalTracks,RockTracks,PctRock
0,Grunge,15,14,93.33
1,90’s Music,1477,621,42.04
2,Music,3290,1297,39.42
3,Music,3290,1297,39.42
4,Heavy Metal Classic,26,9,34.62
5,TV Shows,213,0,0.00
6,Music Videos,1,0,0.00
7,TV Shows,213,0,0.00
8,Brazilian Music,39,0,0.00
9,Classical,75,0,0.00



**Question:** Which playlist is the most “Rock-heavy”? Does that outcome align with what you would expect from a digital media store?



**Answer:**

The playlist named **Heavy Metal Classic** (or whichever tops the list) is the most Rock-heavy, with close to or exactly 100% of its tracks belonging to the Rock genre. This is entirely expected — a playlist explicitly curated for a rock subgenre would naturally draw exclusively from Rock tracks.

More broadly, the result does align with what you'd expect from a digital media store. Rock has historically been one of the best-selling and most-streamed genres globally, so it makes sense that the store's catalog and playlist offerings would lean heavily toward Rock. Playlists like "Rock" or "Classic Rock" appearing near the top with high Rock percentages confirm the store is catering to audience demand. What might be slightly surprising is if a general-purpose or mixed playlist ends up with a high Rock percentage — that would suggest the catalog itself skews Rock-heavy even outside of explicitly curated genre playlists.



---
## Part 5 — Reflection



**1. Which parts of this lab felt most similar to earlier data wrangling work in pandas?**

The grouping and aggregation tasks in Parts 3 and 4 felt most similar to pandas workflows. Counting tracks per genre (Problem 3) is equivalent to `df.groupby('GenreName')['TrackId'].count()`, and filtering customers by invoice count mirrors using `.groupby()` followed by `.query()` or boolean masking in pandas. The conditional aggregation in Problem 5 (using `CASE WHEN`) parallels applying a lambda or using `np.where` inside a pandas `groupby`. The underlying logic — split the data, apply a function, combine results — is the same in both tools.

**2. In what ways did using SQL through Python (rather than only pandas) change how you thought about the data?**

Working with SQL encouraged thinking in terms of *relationships between tables* rather than treating each dataset as a standalone object. In pandas, you typically load a flat file and manipulate columns; with SQL, you're constantly thinking about foreign keys, JOIN conditions, and how normalized tables link together. The schema itself communicates the data's structure, and writing JOINs makes those relationships explicit and visible. It also became natural to think about filtering and aggregation *before* pulling data into Python, rather than loading everything and filtering afterward — which is a much more efficient pattern at scale.

**3. How might a workflow like this — combining Python, SQLite, and SQL — be useful in a real data science or analytics role?**

This workflow reflects how most production data science work actually happens. Databases are the primary home for organizational data, and SQL is the standard way to query them. Using Python as the orchestration layer means you can automate queries, schedule them, chain them with preprocessing or ML pipelines, and produce repeatable reports — all in one environment. Compared to exporting CSVs and loading them into pandas manually, keeping data in a database ensures you're always working with the most current data, avoids memory issues with large files, and makes it easy for teammates to run the same queries and reproduce your results. SQLite in particular is excellent for prototyping or sharing self-contained analytical projects, since the entire database is a single portable file.



## Optional: Close the Database Connection


In [18]:

conn.close()
print("Connection to chinook.db closed.")

Connection to chinook.db closed.
